# v10.2: Curvature-Stabilized Contextual Manifold

## Three Targeted Fixes from the v10.1 Mathematical Audit

v10.1 established that the contextual manifold paradigm works (kappa > 0) but suffers
from **curvature collapse** (kappa: 0.19 → 0.03). The Mathematical Framework notebook
(Section 8) identified the precise gradient-level mechanism and the learning hierarchy
that enables it. v10.2 applies three surgical fixes:

### Fix 1: Detach Curvature from Beta and Routing (breaks the collapse gradient)

The audit found that `d(beta_eff)/d(kappa) < 0` always, creating systematic
gradient pressure toward flatness. Fix: `curvature.detach()` in both the
beta modulation and atom mask computation. The curvature still affects the
architecture's behavior, but the optimizer can no longer flatten the manifold
as an optimization shortcut.

### Fix 2: Curvature Regularizer (actively encourages geometry)

New loss term: `L_curv = -lambda_curv * mean(kappa_t)`. The negative sign means
minimizing the total loss MAXIMIZES curvature. This directly counteracts any
remaining implicit pressure toward flatness. The non-detached curvature is used
here so the gradient flows through to the parallel scan parameters (W_A, W_B, W_psi).

### Fix 3: Attention-Based Manifold Refiner (self-synthesizing geometry)

The v10.1 ManifoldRefiner used a simple MLP to refine q_t from settled representations.
This provides no cross-position information — each position's refinement is independent.

v10.2 replaces this with a multi-head causal attention refiner. Attention on settled
representations computes a context-aware manifold update. This is the Gemini README's
"self-synthesizing geometry" — the manifold is built by the same mechanism (attention)
that gauge theory identifies as the connection.

**Crucially: attention shapes the LANDSCAPE, not the settling process.** The Langevin
SDE remains pure (single force). Attention is NOT inside the settling loop.

### Bonus Fix: Learning Rate Hierarchy

The audit showed manifold parameters (W_A, W_B, W_psi) have the longest gradient path
(O(T) BPTT) and learn slowest. Fix: 3x higher learning rate for manifold parameters.

| Fix | What It Addresses | Mechanism |
|---|---|---|
| Detach curvature | Curvature collapse gradient | Blocks d(beta)/d(kappa) path |
| Curvature regularizer | Manifold goes flat | Gradient reward for curvature |
| Attention refiner | No cross-position manifold info | Self-synthesizing geometry |
| LR hierarchy | Manifold learns too slow | 3x LR for manifold params |

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from typing import Optional, Tuple, Dict, List
from dataclasses import dataclass
from collections import Counter
import math
import os
import urllib.request
from tqdm.notebook import tqdm

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

data_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "v8_real_text", "data")
data_path = os.path.join(data_dir, "input.txt")

if not os.path.exists(data_path):
    os.makedirs(data_dir, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r") as f:
    text = f.read()

def encode(s): return [ord(c) for c in s]
def decode(t): return "".join(chr(x) for x in t if 0 <= x < 256)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"Train: {len(train_data):,} | Val: {len(val_data):,} chars")

Device: cpu
Train: 1,003,854 | Val: 111,540 chars


In [ ]:
@dataclass
class ArchitectureConfig:
    # v10.2: Curvature-stabilized contextual manifold
    # Changes from v10.1: lambda_curv, manifold_lr_scale, n_attn_heads

    fiber_dim: int = 512       # 2x wider representation (THE key scale-up)
    n_subbundles: int = 8      # keeps subbundle_dim = 64 (2x richer per sub)
    manifold_dim: int = 256    # 2:1 compression of fiber (2x more context capacity)

    vocab_size: int = 256
    max_seq_len: int = 128

    atoms_per_subbundle: int = 192  # 50% more attractors per subbundle
    k_wta_min: int = 8
    k_wta_max: int = 32       # more atoms accessible at high curvature

    n_blocks: int = 4
    n_attn_heads: int = 4          # NEW: for attention-based manifold refiner
    langevin_steps: int = 6
    langevin_lr: float = 0.1
    beta_init: float = 1.0
    beta_final: float = 10.0

    sparsity_lambda: float = 0.01
    curvature_beta_scale: float = 0.5
    lambda_curv: float = 0.1       # NEW: curvature regularizer weight
    kappa_target: float = 1.0      # NEW: target curvature (penalize both collapse AND explosion)
    manifold_lr_scale: float = 3.0 # NEW: higher LR for manifold params

    learning_rate: float = 3e-4
    dropout: float = 0.1
    batch_size: int = 16
    seq_len: int = 64
    max_steps: int = 10000
    eval_interval: int = 200
    eval_steps: int = 20

    @property
    def subbundle_dim(self):
        assert self.fiber_dim % self.n_subbundles == 0
        return self.fiber_dim // self.n_subbundles

config = ArchitectureConfig()
print(f"Fiber: {config.fiber_dim} = {config.n_subbundles} x {config.subbundle_dim}")
print(f"Manifold: {config.manifold_dim}d | Attn heads: {config.n_attn_heads}")
print(f"Memory: k in [{config.k_wta_min}, {config.k_wta_max}] (curvature-aware, DETACHED)")
print(f"Curvature reg: lambda={config.lambda_curv}")
print(f"Manifold LR: {config.learning_rate * config.manifold_lr_scale:.1e} ({config.manifold_lr_scale}x base)")

def get_batch(split_data, cfg):
    max_start = len(split_data) - cfg.seq_len - 1
    starts = torch.randint(0, max_start, (cfg.batch_size,))
    return torch.stack([split_data[s:s+cfg.seq_len] for s in starts]).to(device)

Fiber: 512 = 8 x 64
Manifold: 256d | Attn heads: 4
Memory: k in [8, 32] (curvature-aware, DETACHED)
Curvature reg: lambda=0.1
Manifold LR: 9.0e-04 (3.0x base)


## Architecture

Changes from v10.1 are marked with `# FIX 1/2/3` comments.

- **FIX 1** (curvature detach): `CurvatureAwareMemoryBank` and `PureLangevinDescent`
  use `curvature.detach()` for routing and beta modulation
- **FIX 2** (curvature regularizer): `ContextualManifoldCLM` returns non-detached
  curvatures for the loss term
- **FIX 3** (attention refiner): `AttentionManifoldRefiner` replaces `ManifoldRefiner`
  with multi-head causal attention for self-synthesizing geometry

In [31]:
def linear_scan(gates, inputs):
    """Sequential associative scan: h_t = gates_t * h_{t-1} + inputs_t."""
    B, T, D = gates.shape
    h_list = [inputs[:, 0]]
    for t in range(1, T):
        h_list.append(gates[:, t] * h_list[-1] + inputs[:, t])
    return torch.stack(h_list, dim=1)


class ContextualManifold(nn.Module):
    """Wilson line as selective SSM. Unchanged from v10.1."""

    def __init__(self, cfg):
        super().__init__()
        self.A_proj = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.B_proj = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.psi_proj = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.norm = nn.LayerNorm(cfg.manifold_dim)
        with torch.no_grad():
            nn.init.uniform_(self.A_proj.bias, -2.0, 2.0)

    def forward(self, x_sparse):
        A = torch.sigmoid(self.A_proj(x_sparse))
        B = torch.sigmoid(self.B_proj(x_sparse))
        psi = self.psi_proj(x_sparse)
        q = linear_scan(A, B * psi)
        return self.norm(q)


class SparseTokenEmbedding(nn.Module):
    """Sparse fiber bundle sections + contextual manifold. Unchanged from v10.1."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)
        self.topk_per_sub = max(1, cfg.subbundle_dim // 4)
        self.manifold = ContextualManifold(cfg)

    def forward(self, token_ids):
        B, T = token_ids.shape
        x = self.embedding(token_ids)
        chunks = x.chunk(self.cfg.n_subbundles, dim=-1)
        sparse_chunks = []
        for c in chunks:
            _, idx = torch.topk(c.abs(), self.topk_per_sub, dim=-1)
            mask = torch.zeros_like(c)
            mask.scatter_(-1, idx, 1.0)
            sparse_chunks.append(c * mask)
        x_sparse = torch.cat(sparse_chunks, dim=-1)
        q = self.manifold(x_sparse)
        return x_sparse, q


class CurvatureAwareMemoryBank(nn.Module):
    """Memory bank with curvature-dependent atom selection.

    FIX 1: curvature.detach() used for routing decisions.
    Non-detached curvature returned for the regularizer.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        sd, K, A = cfg.subbundle_dim, cfg.n_subbundles, cfg.atoms_per_subbundle
        self.dictionaries = nn.ParameterList(
            [nn.Parameter(torch.randn(A, sd) * 0.02) for _ in range(K)]
        )
        self.routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.manifold_dim + sd, A), nn.SiLU(), nn.Linear(A, A)
            ) for _ in range(K)
        ])

    def compute_curvature(self, q):
        B, T, D = q.shape
        if T < 3:
            return torch.full((B, T), 0.5, device=q.device)
        vel = q[:, 1:] - q[:, :-1]
        accel = q[:, 2:] - 2 * q[:, 1:-1] + q[:, :-2]
        accel_norm = accel.norm(dim=-1)
        vel_norm_sq = vel[:, :-1].norm(dim=-1).pow(2).clamp(min=1e-4)
        kappa = (accel_norm / vel_norm_sq).clamp(max=10.0)
        pad_val = kappa[:, :1]
        kappa = torch.cat([pad_val, pad_val, kappa], dim=1)
        return kappa

    def forward(self, q, x):
        cfg = self.cfg
        sd = cfg.subbundle_dim
        B, T = q.shape[0], q.shape[1]

        curvature = self.compute_curvature(q)

        # FIX 1: DETACH curvature for routing — breaks the collapse gradient
        kappa_for_routing = curvature.detach()
        kappa_norm = (kappa_for_routing / kappa_for_routing.max().clamp(min=1e-8)).clamp(0, 1)
        k_per_token = (cfg.k_wta_min + kappa_norm * (cfg.k_wta_max - cfg.k_wta_min)).long()
        k_per_token = k_per_token.clamp(cfg.k_wta_min, cfg.k_wta_max)

        q_flat = q.reshape(B * T, -1)
        x_flat = x.reshape(B * T, -1)
        k_flat = k_per_token.reshape(B * T)
        atom_mask = torch.arange(cfg.k_wta_max, device=q.device).unsqueeze(0) < k_flat.unsqueeze(-1)

        x_chunks = x_flat.split(sd, dim=-1)
        M_list = []
        for chunk, dic, router in zip(x_chunks, self.dictionaries, self.routers):
            D_n = F.normalize(dic, dim=-1)
            logits = router(torch.cat([q_flat, chunk], dim=-1))
            _, idx = torch.topk(logits, cfg.k_wta_max, dim=-1)
            atoms = D_n[idx]
            atoms = atoms * atom_mask.unsqueeze(-1).float()
            M_list.append(atoms)

        # Return NON-detached curvature for the regularizer (FIX 2)
        return M_list, curvature


class PureLangevinDescent(nn.Module):
    """Pure Hopfield gradient. FIX 1: curvature.detach() for beta modulation."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    def hopfield_gradient(self, x_chunk, M_k, beta):
        sim = torch.bmm(M_k, x_chunk.unsqueeze(-1)).squeeze(-1)
        if isinstance(beta, torch.Tensor):
            w = F.softmax(beta.unsqueeze(-1) * sim, dim=-1)
        else:
            w = F.softmax(beta * sim, dim=-1)
        return -torch.bmm(w.unsqueeze(1), M_k).squeeze(1)

    def forward(self, x_seq, M_q_all, curvature=None,
                return_trajectory=False, return_diagnostics=False):
        cfg = self.cfg
        B, T, D = x_seq.shape
        sd = cfg.subbundle_dim
        x = x_seq.clone()
        betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps)
        trajectory = [x.detach().clone()] if return_trajectory else None
        diagnostics = [] if return_diagnostics else None

        for step in range(cfg.langevin_steps):
            beta_base = betas[step].item()

            if curvature is not None:
                # FIX 1: DETACH curvature from beta — blocks d(beta_eff)/d(kappa) path
                kappa = curvature.detach().clamp(0, 5.0)
                beta_eff = beta_base / (1.0 + cfg.curvature_beta_scale * kappa)
                beta_flat = beta_eff.reshape(B * T)
                noise_scale = torch.sqrt(
                    2.0 * cfg.langevin_lr / beta_eff.clamp(min=0.1)
                ).unsqueeze(-1)
                noise = noise_scale * torch.randn_like(x)
            else:
                beta_flat = beta_base
                noise = math.sqrt(2.0 * cfg.langevin_lr / beta_base) * torch.randn_like(x)

            x_flat = x.reshape(B * T, D)
            x_chunks = x_flat.split(sd, dim=-1)
            grad_chunks = [
                self.hopfield_gradient(xc, mk, beta_flat)
                for xc, mk in zip(x_chunks, M_q_all)
            ]
            grad_E = torch.cat(grad_chunks, dim=-1).reshape(B, T, D)
            x = x - cfg.langevin_lr * grad_E + noise

            if step == cfg.langevin_steps - 1 and cfg.sparsity_lambda > 0:
                threshold = cfg.sparsity_lambda * cfg.langevin_lr
                x = torch.sign(x) * F.relu(x.abs() - threshold)

            if return_trajectory:
                trajectory.append(x.detach().clone())
            if return_diagnostics:
                diagnostics.append({
                    'grad_norm': grad_E.detach().norm(dim=-1).mean().item(),
                    'state_norm': x.detach().norm(dim=-1).mean().item(),
                    'beta_mean': beta_flat.mean().item() if isinstance(beta_flat, torch.Tensor) else beta_flat,
                })

        result = {'settled': x}
        if return_trajectory:
            result['trajectory'] = trajectory
        if return_diagnostics:
            result['diagnostics'] = diagnostics
        return result


class AttentionManifoldRefiner(nn.Module):
    """FIX 3: Self-synthesizing geometry via causal attention.

    Attention on settled representations computes a context-aware
    manifold update. The manifold is built by the same mechanism
    (attention) that gauge theory identifies as the connection.

    Attention shapes the LANDSCAPE, not the settling process.
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_attn_heads
        self.head_dim = cfg.fiber_dim // self.n_heads
        self.scale = math.sqrt(self.head_dim)

        self.W_Q = nn.Linear(cfg.fiber_dim, cfg.fiber_dim, bias=False)
        self.W_K = nn.Linear(cfg.fiber_dim, cfg.fiber_dim, bias=False)
        self.W_V = nn.Linear(cfg.fiber_dim, cfg.fiber_dim, bias=False)

        self.proj_to_manifold = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.gate = nn.Sequential(
            nn.Linear(cfg.manifold_dim * 2, cfg.manifold_dim),
            nn.Sigmoid()
        )

    def forward(self, q, x_settled):
        B, T, D = x_settled.shape

        Q = self.W_Q(x_settled).reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.W_K(x_settled).reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.W_V(x_settled).reshape(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        causal_mask = torch.triu(
            torch.ones(T, T, device=x_settled.device) * float('-inf'), diagonal=1
        )
        scores = scores + causal_mask.unsqueeze(0).unsqueeze(0)
        attn = F.softmax(scores, dim=-1)

        context = torch.matmul(attn, V).transpose(1, 2).reshape(B, T, D)
        delta = self.proj_to_manifold(context)
        g = self.gate(torch.cat([q, delta], dim=-1))
        return q + g * delta


class PureContextualBlock(nn.Module):
    """Memory bank -> pure Langevin -> attention manifold refinement -> residual.
    FIX 3: Uses AttentionManifoldRefiner instead of MLP-based ManifoldRefiner.
    """

    def __init__(self, cfg):
        super().__init__()
        self.memory_bank = CurvatureAwareMemoryBank(cfg)
        self.langevin = PureLangevinDescent(cfg)
        self.refiner = AttentionManifoldRefiner(cfg)  # FIX 3
        self.norm = nn.LayerNorm(cfg.fiber_dim)
        self.dropout = nn.Dropout(cfg.dropout)
        self.residual_gate = nn.Sequential(
            nn.Linear(cfg.fiber_dim * 2, cfg.fiber_dim), nn.Sigmoid()
        )

    def forward(self, x_seq, q, return_diagnostics=False):
        B, T, D = x_seq.shape
        M_q_all, curvature = self.memory_bank(q, x_seq)
        result = self.langevin(
            x_seq, M_q_all, curvature=curvature,
            return_diagnostics=return_diagnostics
        )
        x_settled = result['settled']

        gate_in = torch.cat([x_settled, x_seq], dim=-1)
        gate = self.residual_gate(gate_in.reshape(-1, D * 2)).reshape(B, T, D)
        x_out = self.norm(gate * self.dropout(x_settled) + (1 - gate) * x_seq)

        q_refined = self.refiner(q, x_out)

        output = {'x': x_out, 'q': q_refined, 'curvature': curvature}
        if return_diagnostics:
            output['langevin_diagnostics'] = result.get('diagnostics', [])
        return output


class ContextualManifoldCLM(nn.Module):
    """v10.2: Curvature-stabilized contextual manifold CLM.

    FIX 2: Returns NON-detached curvatures for the regularizer.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = SparseTokenEmbedding(cfg)
        self.blocks = nn.ModuleList([PureContextualBlock(cfg) for _ in range(cfg.n_blocks)])
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.SiLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )
        self.register_buffer("block_loss_weights", torch.linspace(0.1, 1.0, cfg.n_blocks))

    def forward(self, token_ids, return_manifold=False, return_diagnostics=False):
        B, T = token_ids.shape
        x, q = self.embedding(token_ids)

        intermediate_logits = []
        curvatures = []
        block_diagnostics = []

        for block in self.blocks:
            result = block(x, q, return_diagnostics=return_diagnostics)
            x = result['x']
            q = result['q']
            # FIX 2: NON-detached curvature for regularizer gradient
            curvatures.append(result['curvature'])
            intermediate_logits.append(self.decoder(x)[:, :-1, :])
            if return_diagnostics:
                block_diagnostics.append(result.get('langevin_diagnostics', []))

        info = {
            "mean_sparsity": (x.abs() < 1e-3).float().mean().item(),
            "intermediate_logits": intermediate_logits,
            "block_loss_weights": self.block_loss_weights,
            "curvatures": curvatures,
        }
        if return_manifold:
            info["manifold_coords"] = q
            info["final_repr"] = x
        if return_diagnostics:
            info["block_diagnostics"] = block_diagnostics
        return intermediate_logits[-1], info


model = ContextualManifoldCLM(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_manifold = sum(p.numel() for p in model.embedding.manifold.parameters())
n_memory = sum(sum(p.numel() for p in blk.memory_bank.parameters()) for blk in model.blocks)
n_refiner = sum(sum(p.numel() for p in blk.refiner.parameters()) for blk in model.blocks)

print(f"Total parameters: {n_params:,}")
print(f"  Contextual manifold:  {n_manifold:,} ({100*n_manifold/n_params:.1f}%)")
print(f"  Memory banks (4 blk): {n_memory:,} ({100*n_memory/n_params:.1f}%)")
print(f"  Attn refiners (4 blk):{n_refiner:,} ({100*n_refiner/n_params:.1f}%) [NEW: attention-based]")
print(f"  Settling forces:      0 (pure Hopfield gradient)")
print(f"\nv10.1 was 2,471,552 params | v9 was 3,818,672 params")

Total parameters: 10,770,432
  Contextual manifold:  394,496 (3.7%)
  Memory banks (4 blk): 3,551,232 (33.0%)
  Attn refiners (4 blk):4,196,352 (39.0%) [NEW: attention-based]
  Settling forces:      0 (pure Hopfield gradient)

v10.1 was 2,471,552 params | v9 was 3,818,672 params


In [32]:
@torch.no_grad()
def estimate_loss(model, cfg):
    model.eval()
    results = {}
    for name, sd in [("train", train_data), ("val", val_data)]:
        tot_ce, tot_ok, tot_n, tot_sp = 0., 0, 0, 0.
        bces = [0.] * cfg.n_blocks
        curvature_means, curvature_maxs, manifold_vars = [], [], []
        for _ in range(cfg.eval_steps):
            b = get_batch(sd, cfg)
            logits, info = model(b, return_manifold=True)
            tgt = b[:, 1:]
            ce = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
            tot_sp += info["mean_sparsity"]
            for i, bl in enumerate(info["intermediate_logits"]):
                bces[i] += F.cross_entropy(bl.reshape(-1, cfg.vocab_size), tgt.reshape(-1)).item()
            if info["curvatures"]:
                last_curv = info["curvatures"][-1]
                curvature_means.append(last_curv.mean().item())
                curvature_maxs.append(last_curv.max().item())
            if "manifold_coords" in info:
                manifold_vars.append(info["manifold_coords"].var(dim=1).mean().item())
        n = cfg.eval_steps
        results[name] = {
            "ce": tot_ce/n, "acc": tot_ok/tot_n, "sparsity": tot_sp/n,
            "block_ces": [c/n for c in bces],
            "curvature_mean": np.mean(curvature_means) if curvature_means else 0,
            "curvature_max": np.mean(curvature_maxs) if curvature_maxs else 0,
            "manifold_var": np.mean(manifold_vars) if manifold_vars else 0,
        }
    model.train()
    return results


# BONUS FIX: Separate optimizer groups with higher LR for manifold params
manifold_params = list(model.embedding.manifold.parameters())
manifold_param_ids = {id(p) for p in manifold_params}
other_params = [p for p in model.parameters() if id(p) not in manifold_param_ids]

optimizer = torch.optim.AdamW([
    {'params': manifold_params, 'lr': config.learning_rate * config.manifold_lr_scale},
    {'params': other_params, 'lr': config.learning_rate},
], weight_decay=0.01)

history = {
    "step": [], "train_ce": [], "val_ce": [], "train_acc": [], "val_acc": [],
    "train_sparsity": [], "val_sparsity": [],
    "train_bpc": [], "val_bpc": [],
    "block_val_ces": [[] for _ in range(config.n_blocks)],
    "curvature_mean": [], "curvature_max": [],
    "manifold_var": [], "curv_reg_loss": [],
}

print("Training v10.2 (curvature-stabilized contextual manifold)")
print(f"Steps: {config.max_steps}, Batch: {config.batch_size}, Seq: {config.seq_len}")
print(f"FIX 1: curvature.detach() for routing/beta")
print(f"FIX 2: curvature regularizer (lambda={config.lambda_curv})")
print(f"FIX 3: attention-based manifold refiner ({config.n_attn_heads} heads)")
print(f"BONUS: manifold LR = {config.learning_rate * config.manifold_lr_scale:.1e}")
print(f"v10.1 baseline: Val BPC 3.45 | Val Acc 30% | kappa collapsed to 0.03")
print(f"v9 baseline:    Val BPC 2.65 | Val Acc 45%")
print("=" * 70)

model.train()
pbar = tqdm(range(config.max_steps), desc="Training", unit="step")

for step in pbar:
    if step % config.eval_interval == 0:
        res = estimate_loss(model, config)
        tr, vl = res["train"], res["val"]
        history["step"].append(step)
        for k in ["ce", "acc", "sparsity"]:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(vl[k])
        history["train_bpc"].append(tr["ce"] / math.log(2))
        history["val_bpc"].append(vl["ce"] / math.log(2))
        history["curvature_mean"].append(vl["curvature_mean"])
        history["curvature_max"].append(vl["curvature_max"])
        history["manifold_var"].append(vl["manifold_var"])
        for i in range(config.n_blocks):
            history["block_val_ces"][i].append(vl["block_ces"][i])
        tqdm.write(
            f"Step {step:5d} | Train CE: {tr['ce']:.3f} | Val CE: {vl['ce']:.3f} | "
            f"Val BPC: {vl['ce']/math.log(2):.2f} | Val Acc: {vl['acc']:.1%} | "
            f"Sp: {vl['sparsity']:.1%} | kappa: {vl['curvature_mean']:.3f}"
        )

    batch = get_batch(train_data, config)
    optimizer.zero_grad()
    logits, info = model(batch)
    targets = batch[:, 1:]

    # Deep supervision
    ce_loss = sum(
        w * F.cross_entropy(bl.reshape(-1, config.vocab_size), targets.reshape(-1))
        for bl, w in zip(info["intermediate_logits"], info["block_loss_weights"])
    ) / info["block_loss_weights"].sum()

    # Dictionary coherence
    dcl, nd = 0., 0
    for blk in model.blocks:
        for d in blk.memory_bank.dictionaries:
            Dn = F.normalize(d, dim=-1)
            g = Dn @ Dn.T
            dcl += (g - torch.eye(g.size(0), device=g.device)).pow(2).mean()
            nd += 1
    dcl /= max(nd, 1)

    # FIX 2: Curvature regularizer — target range, not unbounded maximization
    # Penalizes deviation from kappa_target in BOTH directions:
    #   too low (collapse) AND too high (explosion)
    # Clamp before squaring to prevent gradient from degenerate curvature values
    curv_reg = 0.0
    for curv in info["curvatures"]:
        clamped = curv.clamp(0, 10.0)
        curv_reg += (clamped.mean() - config.kappa_target).pow(2)
    curv_reg /= len(info["curvatures"])

    loss = ce_loss + 0.1 * dcl + config.lambda_curv * curv_reg
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # Per-step tqdm metrics from the training batch
    with torch.no_grad():
        batch_acc = (logits.detach().argmax(-1) == targets).float().mean().item()
        batch_kappa = info["curvatures"][-1].mean().item()
    pbar.set_postfix(
        ce=f"{ce_loss.item():.3f}",
        acc=f"{batch_acc:.1%}",
        bpc=f"{ce_loss.item()/math.log(2):.2f}",
        kap=f"{batch_kappa:.3f}",
    )

    if step % config.eval_interval == 0:
        history["curv_reg_loss"].append(curv_reg.item())

# Final eval
res = estimate_loss(model, config)
tr, vl = res["train"], res["val"]
history["step"].append(config.max_steps)
for k in ["ce", "acc", "sparsity"]:
    history[f"train_{k}"].append(tr[k])
    history[f"val_{k}"].append(vl[k])
history["train_bpc"].append(tr["ce"] / math.log(2))
history["val_bpc"].append(vl["ce"] / math.log(2))
history["curvature_mean"].append(vl["curvature_mean"])
history["curvature_max"].append(vl["curvature_max"])
history["manifold_var"].append(vl["manifold_var"])
for i in range(config.n_blocks):
    history["block_val_ces"][i].append(vl["block_ces"][i])
history["curv_reg_loss"].append(0.0)

print("=" * 70)
print(f"FINAL | Val CE: {vl['ce']:.3f} | Val BPC: {vl['ce']/math.log(2):.2f} | "
      f"Val Acc: {vl['acc']:.1%} | Val PPL: {math.exp(vl['ce']):.1f}")
print(f"v10.1: Val BPC 3.45 | Val Acc 30% | kappa 0.03 (collapsed)")
print(f"v9:    Val BPC 2.65 | Val Acc 45%")
print(f"Final kappa: {vl['curvature_mean']:.3f} (v10.1 collapsed to 0.03)")

Training v10.2 (curvature-stabilized contextual manifold)
Steps: 10000, Batch: 16, Seq: 64
FIX 1: curvature.detach() for routing/beta
FIX 2: curvature regularizer (lambda=0.1)
FIX 3: attention-based manifold refiner (4 heads)
BONUS: manifold LR = 9.0e-04
v10.1 baseline: Val BPC 3.45 | Val Acc 30% | kappa collapsed to 0.03
v9 baseline:    Val BPC 2.65 | Val Acc 45%


Training:   0%|          | 0/10000 [00:00<?, ?step/s]

Step     0 | Train CE: 5.577 | Val CE: 5.574 | Val BPC: 8.04 | Val Acc: 0.3% | Sp: 0.1% | kappa: 0.133
Step   200 | Train CE: 2.492 | Val CE: 2.504 | Val BPC: 3.61 | Val Acc: 27.4% | Sp: 0.1% | kappa: 0.297
Step   400 | Train CE: 2.460 | Val CE: 2.484 | Val BPC: 3.58 | Val Acc: 27.9% | Sp: 0.2% | kappa: 0.302
Step   600 | Train CE: 2.406 | Val CE: 2.453 | Val BPC: 3.54 | Val Acc: 28.9% | Sp: 0.2% | kappa: 0.294


KeyboardInterrupt: 

In [ ]:
steps = history["step"]
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

axes[0,0].plot(steps, history["train_ce"], 'b-', label='Train')
axes[0,0].plot(steps, history["val_ce"], 'r-', label='Val')
axes[0,0].set_title('Cross-Entropy'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(steps, history["train_bpc"], 'b-', label='Train')
axes[0,1].plot(steps, history["val_bpc"], 'r-', label='Val')
axes[0,1].axhline(y=2.65, color='gray', linestyle='--', alpha=0.7, label='v9 (2.65)')
axes[0,1].axhline(y=3.45, color='orange', linestyle=':', alpha=0.7, label='v10.1 (3.45)')
axes[0,1].set_title('Bits per Character'); axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(steps, history["train_acc"], 'b-', label='Train')
axes[0,2].plot(steps, history["val_acc"], 'r-', label='Val')
axes[0,2].axhline(y=0.45, color='gray', linestyle='--', alpha=0.7, label='v9 (45%)')
axes[0,2].axhline(y=0.30, color='orange', linestyle=':', alpha=0.7, label='v10.1 (30%)')
axes[0,2].set_title('Accuracy'); axes[0,2].legend(fontsize=8); axes[0,2].grid(True, alpha=0.3)

gap = [v - t for v, t in zip(history["val_ce"], history["train_ce"])]
axes[1,0].plot(steps, gap, 'purple', lw=2)
axes[1,0].set_title('Generalization Gap'); axes[1,0].grid(True, alpha=0.3)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, config.n_blocks))
for i in range(config.n_blocks):
    axes[1,1].plot(steps, history["block_val_ces"][i], color=colors[i], label=f'Block {i}')
axes[1,1].set_title('Per-Block Val CE'); axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(steps, history["val_sparsity"], 'r-')
axes[1,2].set_title('Output Sparsity'); axes[1,2].grid(True, alpha=0.3)

# Row 3: THE KEY QUESTION — does curvature stay stable?
axes[2,0].plot(steps, history["curvature_mean"], 'teal', lw=2.5, label='v10.2 mean kappa')
axes[2,0].axhline(y=0.03, color='red', linestyle='--', lw=2, alpha=0.7, label='v10.1 collapsed to 0.03')
axes[2,0].set_title('CURVATURE STABILITY\n(Did FIX 1+2 prevent collapse?)')
axes[2,0].set_xlabel('Step'); axes[2,0].legend(fontsize=9); axes[2,0].grid(True, alpha=0.3)

axes[2,1].plot(steps, history["manifold_var"], 'steelblue', lw=2)
axes[2,1].set_title('Manifold Coordinate Variance')
axes[2,1].set_xlabel('Step'); axes[2,1].grid(True, alpha=0.3)

n_curv = min(len(steps), len(history["curv_reg_loss"]))
if n_curv > 0:
    axes[2,2].plot(steps[:n_curv], history["curv_reg_loss"][:n_curv], 'coral', lw=2)
    axes[2,2].set_title('Curvature Regularizer Loss\n(more negative = higher curvature)')
    axes[2,2].set_xlabel('Step'); axes[2,2].grid(True, alpha=0.3)

plt.suptitle('v10.2: Curvature-Stabilized Contextual Manifold — Training Diagnostics',
             fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nFinal: Val BPC {history['val_bpc'][-1]:.2f} | Val Acc {history['val_acc'][-1]:.1%}")
print(f"v10.1: Val BPC 3.45 | Val Acc 30% | kappa 0.03")
print(f"v9:    Val BPC 2.65 | Val Acc 45%")
delta_bpc_v101 = history['val_bpc'][-1] - 3.45
delta_acc_v101 = (history['val_acc'][-1] - 0.30) * 100
print(f"Delta vs v10.1: BPC {delta_bpc_v101:+.2f} | Acc {delta_acc_v101:+.1f}pp")
print(f"Final kappa: {history['curvature_mean'][-1]:.3f} (v10.1 collapsed to 0.03)")

In [ ]:
@torch.no_grad()
def generate_text(model, prompt_str, max_new=200, temperature=0.8):
    model.eval()
    cfg = model.cfg
    ids = torch.tensor(encode(prompt_str), dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new):
        ctx = ids[:, -cfg.max_seq_len:] if ids.size(1) >= cfg.max_seq_len else ids
        logits, _ = model(ctx)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        ids = torch.cat([ids, torch.multinomial(probs, 1)], dim=1)
    return decode(ids[0].tolist())

print("=" * 60)
print("TEXT GENERATION (v10.2, temperature=0.8)")
print("=" * 60)
for p in ["ROMEO:\n", "To be or not to ", "The king ", "O, "]:
    print(f"\n--- Prompt: {repr(p)} ---")
    print(generate_text(model, p))

In [ ]:
@torch.no_grad()
def test_manifold_contextuality(model, cfg):
    """Tests 1-3: Is the manifold contextual AND does curvature survive training?"""
    model.eval()
    text_a = "ROMEO:\nO, she doth teach the torches to burn bright!"
    text_b = "JULIET:\nWhat's in a name? That which we call a rose"
    T = min(cfg.seq_len, len(text_a), len(text_b))
    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)

    _, info_a = model(ids_a, return_manifold=True)
    _, info_b = model(ids_b, return_manifold=True)
    q_a = info_a["manifold_coords"][0].cpu()
    q_b = info_b["manifold_coords"][0].cpu()

    divergence = (q_a - q_b).norm(dim=-1).numpy()
    combined = torch.cat([q_a, q_b], dim=0)
    centered = combined - combined.mean(dim=0, keepdim=True)
    U, S, Vh = torch.linalg.svd(centered, full_matrices=False)
    explained = (S[:3] ** 2) / (S ** 2).sum()
    proj = (centered @ Vh.T[:, :3]).numpy()
    q_a_3d, q_b_3d = proj[:T], proj[T:]

    x_a, q_a_t = model.embedding(ids_a)
    x_b, q_b_t = model.embedding(ids_b)
    block = model.blocks[0]
    jaccards = []
    for t in range(T):
        sub_jacs = []
        for k, router in enumerate(block.memory_bank.routers):
            sd = cfg.subbundle_dim
            chunk_a = x_a[0, t:t+1, k*sd:(k+1)*sd]
            chunk_b = x_b[0, t:t+1, k*sd:(k+1)*sd]
            logits_a = router(torch.cat([q_a_t[0, t:t+1], chunk_a], dim=-1))
            logits_b = router(torch.cat([q_b_t[0, t:t+1], chunk_b], dim=-1))
            _, idx_a = torch.topk(logits_a, cfg.k_wta_max, dim=-1)
            _, idx_b = torch.topk(logits_b, cfg.k_wta_max, dim=-1)
            set_a = set(idx_a[0].cpu().numpy().tolist())
            set_b = set(idx_b[0].cpu().numpy().tolist())
            jac = len(set_a & set_b) / len(set_a | set_b) if set_a | set_b else 1.0
            sub_jacs.append(jac)
        jaccards.append(np.mean(sub_jacs))

    fig = plt.figure(figsize=(18, 5))
    ax1 = fig.add_subplot(131)
    ax1.plot(divergence, 'b-', lw=2)
    ax1.set_xlabel('Position'); ax1.set_ylabel('||q_a - q_b||')
    ax1.set_title('Test 1: Manifold Divergence')
    ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='v1-v9 (0.0)')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(132, projection='3d')
    ax2.plot(*q_a_3d.T, 'b-o', ms=3, label='ROMEO...')
    ax2.plot(*q_b_3d.T, 'r-o', ms=3, label='JULIET...')
    ax2.set_title(f'Test 2: Manifold Trajectories\n{explained.sum():.0%} var in 3D')
    ax2.legend(fontsize=8)

    ax3 = fig.add_subplot(133)
    ax3.plot(jaccards, 'g-', lw=2)
    ax3.set_xlabel('Position'); ax3.set_ylabel('Jaccard')
    ax3.set_title('Test 3: Atom Overlap')
    ax3.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='v1-v9 (1.0)')
    ax3.set_ylim(0, 1.1); ax3.legend(); ax3.grid(True, alpha=0.3)

    plt.suptitle('v10.2 Diagnostics: Manifold Contextuality (Tests 1-3)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f"\nTest 1: Mean divergence = {divergence.mean():.4f}")
    print(f"Test 3: Mean Jaccard   = {np.mean(jaccards):.4f}")

test_manifold_contextuality(model, config)

In [ ]:
@torch.no_grad()
def test_context_and_curvature(model, cfg):
    """Tests 4-5: Context accumulation. Tests 6-7: Pure SDE + curvature."""
    model.eval()

    # Test 4: Probe q_t
    n_samples = 2000
    qs, targets = [], []
    for _ in range(n_samples // (cfg.batch_size * cfg.seq_len) + 1):
        batch = get_batch(val_data, cfg)
        _, info = model(batch, return_manifold=True)
        q = info["manifold_coords"][:, 1:, :]
        tgt = batch[:, :-1]
        qs.append(q.reshape(-1, cfg.manifold_dim).cpu())
        targets.append(tgt.reshape(-1).cpu())
    Q = torch.cat(qs, dim=0)[:n_samples]
    Y = torch.cat(targets, dim=0)[:n_samples]
    counts = torch.bincount(Y, minlength=256)
    top_chars = counts.topk(30).indices
    mask = torch.zeros(len(Y), dtype=torch.bool)
    for c in top_chars:
        mask |= (Y == c)
    Q_sub, Y_sub = Q[mask], Y[mask]
    n_train = int(0.8 * len(Q_sub))
    Y_onehot = F.one_hot(Y_sub[:n_train].long(), num_classes=256).float()
    W_probe = torch.linalg.lstsq(Q_sub[:n_train], Y_onehot).solution
    preds = (Q_sub[n_train:] @ W_probe).argmax(dim=-1)
    probe_acc = (preds == Y_sub[n_train:]).float().mean().item()
    baseline_acc = torch.bincount(Y_sub[n_train:].long(), minlength=256).max().item() / len(Y_sub[n_train:])

    # Test 6: Context-sensitive Hopfield gradient
    text_a = "The brave knight drew his sword and charged"
    text_b = "The quiet child drew his picture and smiled"
    T = min(cfg.seq_len, len(text_a), len(text_b))
    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)
    x_a, q_a = model.embedding(ids_a)
    x_b, q_b = model.embedding(ids_b)
    M_a, _ = model.blocks[0].memory_bank(q_a, x_a)
    M_b, _ = model.blocks[0].memory_bank(q_b, x_b)
    sd = cfg.subbundle_dim
    grad_a_all, grad_b_all = [], []
    for k in range(cfg.n_subbundles):
        xa_sub = x_a.reshape(-1, cfg.fiber_dim)[:, k*sd:(k+1)*sd]
        xb_sub = x_b.reshape(-1, cfg.fiber_dim)[:, k*sd:(k+1)*sd]
        wa = F.softmax(5.0 * torch.bmm(M_a[k], xa_sub.unsqueeze(-1)).squeeze(-1), dim=-1)
        wb = F.softmax(5.0 * torch.bmm(M_b[k], xb_sub.unsqueeze(-1)).squeeze(-1), dim=-1)
        grad_a_all.append(-torch.bmm(wa.unsqueeze(1), M_a[k]).squeeze(1))
        grad_b_all.append(-torch.bmm(wb.unsqueeze(1), M_b[k]).squeeze(1))
    grad_a = torch.cat(grad_a_all, dim=-1).reshape(1, T, cfg.fiber_dim)
    grad_b = torch.cat(grad_b_all, dim=-1).reshape(1, T, cfg.fiber_dim)
    cos_sim = F.cosine_similarity(grad_a[0], grad_b[0], dim=-1).cpu().numpy()

    # Test 7: Curvature-dependent exploration
    batch = get_batch(val_data, cfg)
    x, q = model.embedding(batch)
    M_list, curvature = model.blocks[0].memory_bank(q, x)
    total_active = torch.zeros(cfg.batch_size, cfg.seq_len, device=device)
    for M_k in M_list:
        norms = M_k.reshape(cfg.batch_size, cfg.seq_len, cfg.k_wta_max, -1).norm(dim=-1)
        total_active += (norms > 1e-6).float().sum(dim=-1)
    avg_active = (total_active / cfg.n_subbundles).reshape(-1).cpu().numpy()
    curv_flat = curvature.reshape(-1).cpu().numpy()

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    bars = axes[0,0].bar(['Probe (q_t)', 'Baseline'], [probe_acc, baseline_acc],
                          color=['steelblue', 'lightcoral'])
    axes[0,0].set_title('Test 4: Linear Probe on q_t'); axes[0,0].set_ylim(0, 1)
    for bar, val in zip(bars, [probe_acc, baseline_acc]):
        axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                       f'{val:.1%}', ha='center', fontweight='bold')
    axes[0,0].grid(True, alpha=0.3, axis='y')

    axes[0,1].plot(cos_sim, 'purple', lw=2)
    axes[0,1].axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Context-blind')
    axes[0,1].set_title('Test 6: Hopfield Gradient Context Sensitivity')
    axes[0,1].set_xlabel('Position'); axes[0,1].set_ylim(-0.5, 1.1)
    axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

    axes[1,0].scatter(curv_flat, avg_active, alpha=0.05, s=5, c='teal')
    axes[1,0].set_xlabel('Curvature'); axes[1,0].set_ylabel('Active atoms/sub')
    axes[1,0].set_title('Test 7: Curvature vs Active Atoms')
    axes[1,0].grid(True, alpha=0.3)

    # Test 8: Holonomy
    text = "ROMEO:\nBut, soft! What light through yonder window breaks?"
    T_h = min(cfg.seq_len, len(text))
    ids_orig = torch.tensor(encode(text[:T_h]), dtype=torch.long).unsqueeze(0).to(device)
    holonomies = []
    for pos in range(2, T_h - 2):
        ids_swap = ids_orig.clone()
        ids_swap[0, pos], ids_swap[0, pos+1] = ids_swap[0, pos+1].item(), ids_swap[0, pos].item()
        _, io = model(ids_orig, return_manifold=True)
        _, iw = model(ids_swap, return_manifold=True)
        qo, qw = io["manifold_coords"][0], iw["manifold_coords"][0]
        holonomies.append((qo[pos+2:] - qw[pos+2:]).norm(dim=-1).mean().item())

    axes[1,1].plot(range(2, T_h-2), holonomies, 'b-o', ms=3)
    axes[1,1].set_xlabel('Swap Position'); axes[1,1].set_ylabel('Downstream ||dq||')
    axes[1,1].set_title('Test 8: Small-Loop Holonomy')
    axes[1,1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
    axes[1,1].grid(True, alpha=0.3)

    plt.suptitle('v10.2 Diagnostics: Context, Gradient Sensitivity, Curvature, Holonomy',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    corr = np.corrcoef(curv_flat, avg_active)[0, 1] if len(curv_flat) > 1 else 0
    print(f"Test 4: Probe acc = {probe_acc:.1%} (baseline: {baseline_acc:.1%})")
    print(f"Test 6: Mean gradient cos = {cos_sim.mean():.4f}")
    print(f"Test 7: Corr(curvature, atoms) = {corr:.4f}")
    print(f"Test 8: Mean holonomy = {np.mean(holonomies):.4f}")

test_context_and_curvature(model, config)

In [ ]:
@torch.no_grad()
def test_sparse_patterns(model, cfg):
    """Tests 9-10: Sparse fiber bundle activation patterns."""
    model.eval()
    text_a = "The quality of mercy is not strained, it droppeth as"
    text_b = "The quantity of arrows is not counted, it striketh at"
    T = min(cfg.seq_len, len(text_a), len(text_b))
    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)
    _, info_a = model(ids_a, return_manifold=True)
    _, info_b = model(ids_b, return_manifold=True)
    repr_a = info_a["final_repr"][0].cpu()
    repr_b = info_b["final_repr"][0].cpu()
    sd = cfg.subbundle_dim
    hamming_repr, emd_repr = [], []
    for t in range(T):
        sub_h, sub_e = [], []
        for k in range(cfg.n_subbundles):
            ra = repr_a[t, k*sd:(k+1)*sd]
            rb = repr_b[t, k*sd:(k+1)*sd]
            ma = (ra.abs() > ra.abs().mean() * 0.1).float()
            mb = (rb.abs() > rb.abs().mean() * 0.1).float()
            sub_h.append((ma - mb).abs().sum().item() / sd)
            sub_e.append((ra.abs().sort(descending=True).values - rb.abs().sort(descending=True).values).abs().sum().item())
        hamming_repr.append(np.mean(sub_h))
        emd_repr.append(np.mean(sub_e))

    batch = get_batch(val_data, cfg)
    _, batch_info = model(batch, return_manifold=True)
    batch_repr = batch_info["final_repr"].cpu()
    subbundle_entropies = []
    for k in range(cfg.n_subbundles):
        sub_repr = batch_repr[:, :, k*sd:(k+1)*sd]
        topk = min(4, sd)
        _, top_dims = sub_repr.abs().topk(topk, dim=-1)
        patterns = top_dims.reshape(-1, topk).numpy()
        pattern_strs = [tuple(sorted(p)) for p in patterns]
        counts = Counter(pattern_strs)
        total = sum(counts.values())
        probs = np.array(list(counts.values())) / total
        subbundle_entropies.append(-np.sum(probs * np.log(probs + 1e-10)))
    theoretical_max = np.log(math.comb(sd, min(4, sd)))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(hamming_repr, 'r-', lw=2)
    axes[0].set_xlabel('Position'); axes[0].set_ylabel('Hamming')
    axes[0].set_title('Test 9a: Support Mask Divergence'); axes[0].grid(True, alpha=0.3)
    axes[1].plot(emd_repr, 'purple', lw=2)
    axes[1].set_xlabel('Position'); axes[1].set_ylabel('EMD')
    axes[1].set_title('Test 9b: Activation Pattern Distance'); axes[1].grid(True, alpha=0.3)
    axes[2].bar(range(cfg.n_subbundles), subbundle_entropies, color='teal', alpha=0.8)
    axes[2].axhline(y=theoretical_max, color='red', linestyle='--', alpha=0.5, label=f'Max = {theoretical_max:.1f}')
    axes[2].set_xlabel('Subbundle'); axes[2].set_ylabel('Entropy (nats)')
    axes[2].set_title('Test 10: Pattern Entropy'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.suptitle('v10.2 Diagnostics: Sparse Patterns (Tests 9-10)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f"Test 9: Mean Hamming = {np.mean(hamming_repr):.4f}, Mean EMD = {np.mean(emd_repr):.4f}")
    print(f"Test 10: Mean entropy = {np.mean(subbundle_entropies):.2f} / {theoretical_max:.2f} = {np.mean(subbundle_entropies)/theoretical_max:.0%}")

test_sparse_patterns(model, config)

## Summary

### What v10.2 Tests

The three fixes address the specific pathologies identified by the v10.1 audit:

| Fix | Pathology | Mechanism | Expected Effect |
|---|---|---|---|
| Detach curvature | Collapse gradient d(beta)/d(kappa) | Blocks gradient path | kappa stays stable |
| Curvature regularizer | No incentive to maintain geometry | Gradient reward for curvature | kappa increases |
| Attention refiner | No cross-position manifold info | Self-synthesizing geometry | Better manifold quality |
| LR hierarchy | Manifold learns too slowly | 3x LR for manifold params | Faster manifold adaptation |

### Key Metric to Watch

**Curvature stability:** If kappa stays above 0.1 throughout training (vs v10.1's collapse
to 0.03), Fix 1+2 are working. If accuracy exceeds 30% (v10.1's ceiling), the stabilized
curvature is translating to better representations. If accuracy approaches 45% (v9), the
pure SDE + attention refiner recovers v9's cross-position capability.

### Architecture Comparison

| | v9 | v10.1 | v10.2 |
|---|---|---|---|
| Manifold | Positional | Contextual (scan) | Contextual (scan) |
| Settling forces | 4 (Hopfield+conv+inhib+attn) | 1 (Hopfield only) | 1 (Hopfield only) |
| Cross-position | Attention inside Langevin | None | Attention in refiner |
| Curvature | Not computed | Collapses to 0.03 | Stabilized by detach + reg |
| Manifold LR | Same as all params | Same as all params | 3x higher |
| Params | ~3.8M | ~2.5M | ~3.2M |

### If This Works

The pure SDE hypothesis is validated *with structural support*: the context IS the
landscape, but the landscape needs active protection from optimization pressure.
Attention builds the geometry (self-synthesizing), not the settling dynamics.

### If This Doesn't Work

The curvature-modulated routing and beta may need to be removed entirely (use fixed k
and fixed beta schedule). The manifold's value may come purely from context-aware router
inputs, not from curvature-dependent modulation. Simplify further.

---

*v10.2: Curvature-stabilized contextual manifold with self-synthesizing geometry.
Built from the Mathematical Framework audit of v10.1.*